# Módulo 02 · Aula 01 — Fundamentos do Git

> **Manual de Estudos Interativo** · Trilha Engenharia de Software & Dados
> Projeto transversal: **Atlas / Aurora Comércio**

## A dor da Aurora

> *"Ontem o estagiário salvou por cima do `relatorio_vendas.py`. A gente tinha a versão que funcionava… agora tem `relatorio_final.py`, `relatorio_final_v2.py`, `relatorio_final_AGORA_VAI.py` e nenhuma delas roda. Perdemos o dia."*
> — Você, na sua segunda semana

Versionar não é sobre "backup". É sobre poder responder, a qualquer momento:

- **O que** mudou?
- **Quem** mudou?
- **Quando** mudou?
- **Por que** mudou?
- Como eu **volto** para antes de tudo dar errado?

## O que você vai aprender aqui

| # | Tópico | Por que importa |
|---|--------|-----------------|
| 1 | Modelo mental do Git | Sem isso, os comandos viram decoreba |
| 2 | Instalação e `git config` | Configurar uma vez, usar sempre |
| 3 | `init` e as três áreas | Working directory, staging, repositório |
| 4 | `status`, `add`, `commit` | O ciclo básico, 90% do uso diário |
| 5 | `log` e suas variações | Ler a história do projeto |
| 6 | `diff` | Ver exatamente o que mudou |
| 7 | `.gitignore` | O que **não** versionar |
| 8 | Mensagens de commit | O que separa amador de profissional |

## ⚙️ Como este notebook funciona

Git é uma ferramenta de **terminal**. Para praticar sem sair do notebook, usamos o prefixo `!`, que executa um comando no shell do sistema:

```python
!git status
```

**Um detalhe importante:** cada `!` abre um shell novo, então `!cd pasta` **não** funciona — a pasta volta ao normal na linha seguinte. Por isso usamos a flag `-C`:

```python
!git -C caminho/do/repo status
```

`git -C <pasta>` significa "execute como se você estivesse dentro de `<pasta>`".

> 💡 **No terminal de verdade** você faria `cd caminho/do/repo` uma vez e depois só `git status`. A flag `-C` é uma muleta do notebook. Ao praticar no seu terminal, esqueça o `-C`.

Execute a célula abaixo para conferir se o Git está instalado.

In [ ]:
!git --version

### Se o comando acima falhou

| Sistema | Como instalar |
|---------|---------------|
| **Windows** | Baixe o [Git for Windows](https://git-scm.com/download/win). Aceite os padrões, exceto: escolha **"Git from the command line and also from 3rd-party software"** e **"Checkout as-is, commit Unix-style line endings"**. |
| **macOS** | `brew install git` (ou instale as Command Line Tools: `xcode-select --install`) |
| **Linux (Debian/Ubuntu)** | `sudo apt update && sudo apt install git` |
| **Linux (Fedora)** | `sudo dnf install git` |

Depois de instalar, **reinicie o VS Code** — ele só enxerga o novo PATH em uma sessão nova.

## 1. O modelo mental: Git guarda *snapshots*, não diferenças

Esta é a ideia que faz tudo o mais fazer sentido.

Muitos sistemas antigos de versionamento guardavam **a lista de mudanças** de cada arquivo ao longo do tempo. O Git faz diferente: a cada commit, ele tira uma **fotografia do projeto inteiro**.

```
              commit A          commit B          commit C
             ┌────────┐        ┌────────┐        ┌────────┐
 main.py     │ v1     │───────▶│ v1  ↺  │───────▶│ v2     │
 utils.py    │ v1     │───────▶│ v2     │───────▶│ v2  ↺  │
 README.md   │ v1     │───────▶│ v1  ↺  │───────▶│ v1  ↺  │
             └────────┘        └────────┘        └────────┘
                                 ↺ = arquivo não mudou:
                                     o Git só guarda um ponteiro
                                     para a versão anterior
```

Quando um arquivo não muda, o Git não duplica: ele guarda uma **referência** ao arquivo já armazenado. É por isso que um repositório com milhares de commits ocupa pouco espaço.

**Consequência prática:** um commit é um estado completo e coerente do projeto. Voltar a um commit é voltar o projeto inteiro àquele instante — não é "desfazer uma mudança".

### As três áreas (e uma quarta)

Este diagrama é o mapa que você vai consultar mentalmente todos os dias:

```
 ┌──────────────────┐   git add    ┌──────────────────┐  git commit  ┌──────────────────┐
 │ Working Directory│─────────────▶│  Staging Area    │─────────────▶│   Repositório    │
 │  (seus arquivos) │              │    (o "índice")  │              │  (.git/ — commits)│
 └──────────────────┘◀─────────────└──────────────────┘◀─────────────└──────────────────┘
                       git restore                       git reset
                        --staged
        │                                                                      │
        └──────────────────── git restore <arquivo> ◀──────────────────────────┘
                             (descarta e volta ao último commit)
```

| Área | O que é | Comando para entrar |
|------|---------|---------------------|
| **Working Directory** | A pasta que você vê e edita | (você editando) |
| **Staging Area** (index) | A "sacola de compras": o que vai no próximo commit | `git add` |
| **Repositório** (`.git/`) | O histórico permanente | `git commit` |
| **Remoto** (GitHub) | Cópia no servidor | `git push` (aula 02) |

### Por que existe a staging area?

É a pergunta que todo iniciante faz. A resposta: ela permite **escolher o que entra em cada commit**.

Você mexeu em 5 arquivos, mas 3 são a correção de um bug e 2 são uma funcionalidade nova. Sem staging, você faria um commit bagunçado. Com staging, você faz `git add` nos 3 primeiros, commita "corrige cálculo de frete", depois `add` nos outros 2 e commita "adiciona relatório por canal".

**Dois commits pequenos e coerentes > um commit gigante.** Sempre.

## 2. Configuração inicial

O Git precisa saber quem você é — esse dado vai gravado em **cada commit**, permanentemente.

```bash
git config --global user.name "Seu Nome"
git config --global user.email "seu@email.com"
```

| Nível | Flag | Onde fica | Vale para |
|-------|------|-----------|-----------|
| Sistema | `--system` | `/etc/gitconfig` | Todos os usuários |
| **Global** | `--global` | `~/.gitconfig` | **Você, em todos os repos** ← use este |
| Local | (nenhuma) | `.git/config` | Só aquele repositório |

O mais específico vence. Se você trabalha com e-mail pessoal e corporativo, configure o global com o pessoal e sobrescreva localmente nos repos do trabalho.

### Configurações que valem a pena

```bash
# Nome do branch inicial (o padrão histórico era 'master')
git config --global init.defaultBranch main

# Editor para mensagens de commit (troque por 'nano', 'vim' etc.)
git config --global core.editor "code --wait"

# Fim de linha — CRÍTICO em times mistos Windows/Linux
git config --global core.autocrlf true    # Windows
git config --global core.autocrlf input   # macOS e Linux

# Ao dar 'git pull', usar rebase em vez de criar commit de merge
git config --global pull.rebase false     # padrão explícito, evita aviso
```

In [ ]:
# Veja sua configuração atual (pode estar vazia se você nunca configurou)
!git config --global --list

> ⚠️ **Se o comando acima deu erro ou veio vazio**, configure agora. Abra o terminal do VS Code (`Ctrl+'`) e rode os dois comandos de `user.name` e `user.email` com seus dados reais. Depois volte aqui.
>
> Você **não consegue** fazer commit sem isso — o Git recusa.

## 3. Criando o laboratório

Vamos criar um repositório de treino descartável. Toda esta aula acontece dentro dele — seu computador não é afetado em mais nada.

A célula abaixo é **idempotente**: pode rodar quantas vezes quiser, sempre recomeça do zero.

In [ ]:
import shutil
import subprocess
from pathlib import Path

LAB = Path("lab_git/repo_treino").resolve()

# Recomeça do zero a cada execução
shutil.rmtree(LAB.parent, ignore_errors=True)
LAB.mkdir(parents=True)

print("📁 Laboratório em:", LAB)
print("   (esta pasta é descartável — apague quando quiser)")


def git(*args, cwd=LAB, mostrar=True):
    """Executa um comando git no laboratório e imprime a saída.

    Existe só para deixar as células curtas. No terminal você digitaria
    o comando direto, sem esta função.
    """
    comando = ["git", *args]
    r = subprocess.run(comando, cwd=cwd, capture_output=True, text=True)
    if mostrar:
        print("$ git " + " ".join(args))
        saida = (r.stdout + r.stderr).rstrip()
        print(saida if saida else "(sem saída)")
        print()
    return r


def escrever(nome, conteudo, pasta=LAB):
    """Cria/sobrescreve um arquivo no laboratório."""
    caminho = Path(pasta) / nome
    caminho.parent.mkdir(parents=True, exist_ok=True)
    caminho.write_text(conteudo, encoding="utf-8")
    print(f"✏️  escrito: {nome} ({len(conteudo)} caracteres)")


print("✅ Funções auxiliares prontas: git(...) e escrever(...)")

## 4. `git init` — criando o repositório

```bash
git init
```

Isso cria uma pasta oculta `.git/` — **ela é o repositório**. Todo o histórico mora ali. Apagar `.git/` transforma o projeto de volta em uma pasta comum, sem histórico.

> ⚠️ Nunca edite nada dentro de `.git/` manualmente.

In [ ]:
git("init", "-b", "main")   # -b main já nomeia o branch inicial

# Identidade LOCAL, só para este laboratório.
# No seu projeto real, use --global uma vez e esqueça.
git("config", "user.name", "Aluno Atlas")
git("config", "user.email", "aluno@aurora.com.br")

In [ ]:
# O que o init criou?
import os

print("Conteúdo de .git/ :")
for item in sorted(os.listdir(LAB / ".git")):
    print("   ", item)

## 5. O ciclo básico: `status` → `add` → `commit`

### `git status` — a bússola

**Rode `git status` o tempo todo.** Sério. Em caso de dúvida sobre qualquer coisa, `git status`. Ele te diz onde você está, o que mudou e o que fazer em seguida.

Estados possíveis de um arquivo:

```
  untracked  ──git add──▶  staged  ──git commit──▶  committed
 (o Git nunca                                       (no histórico)
  viu este arquivo)                                       │
                                                          │ você edita
                                                          ▼
                              staged  ◀──git add──   modified
                                                    (rastreado e alterado)
```

In [ ]:
git("status")

In [ ]:
# Vamos criar o primeiro arquivo do Atlas
escrever("relatorio_vendas.py", '''"""Relatório de vendas da Aurora Comércio."""


def calcular_faturamento(pedidos):
    """Soma o valor dos pedidos pagos."""
    total = 0.0
    for p in pedidos:
        if p["status"] == "pago":
            total += p["quantidade"] * p["preco"]
    return total


if __name__ == "__main__":
    print("Atlas — relatório de vendas")
''')

git("status")

> 📌 Repare em **`Untracked files`**. O Git viu o arquivo, mas não está cuidando dele. Arquivo untracked não entra em commit, não aparece em `diff`, não é protegido. Ele precisa ser **adicionado** primeiro.

In [ ]:
# git add — move para a staging area
git("add", "relatorio_vendas.py")
git("status")

### Variações do `git add`

| Comando | O que faz |
|---------|-----------|
| `git add arquivo.py` | Um arquivo específico |
| `git add pasta/` | Tudo dentro da pasta |
| `git add .` | Tudo a partir da pasta atual |
| `git add -A` | Tudo do repositório, inclusive remoções |
| `git add *.py` | Por padrão de nome |
| `git add -p` | **Interativo**: escolhe pedaço por pedaço |

> ⚠️ **`git add .` é conveniente e perigoso.** Ele pega tudo — inclusive aquele arquivo de senha que você esqueceu de ignorar. Pegue o hábito de rodar `git status` **antes** de qualquer `add .`.
>
> 💡 **`git add -p`** é a ferramenta dos profissionais: ela mostra cada trecho alterado e pergunta se entra no commit. É como você separa uma correção de bug de uma refatoração que ficaram no mesmo arquivo.

In [ ]:
# git commit — grava o snapshot
git("commit", "-m", "feat: cria script de cálculo de faturamento")
git("status")

### Anatomia de um commit

Todo commit tem:

| Campo | Exemplo |
|-------|---------|
| **Hash (SHA-1)** | `a3f5c9e...` — identificador único, 40 caracteres |
| **Autor** | quem escreveu o código |
| **Committer** | quem aplicou (geralmente o mesmo) |
| **Data** | timestamp |
| **Mensagem** | a explicação — a parte que **você** controla |
| **Árvore** | o snapshot dos arquivos |
| **Pai(s)** | o commit anterior |

Na prática você usa os **7 primeiros caracteres** do hash (`a3f5c9e`) — já é suficiente para identificar sem ambiguidade em repositórios normais.

In [ ]:
git("log")

## 6. Mensagens de commit: o que separa amador de profissional

Daqui a seis meses, você vai olhar o histórico procurando **quando** aquele bug entrou. Se todos os commits dizem "ajustes", você está sozinho.

### O padrão *Conventional Commits*

```
<tipo>(<escopo opcional>): <descrição curta no imperativo>

<corpo opcional: POR QUE, não O QUÊ>

<rodapé opcional: refs, breaking changes>
```

| Tipo | Quando usar |
|------|-------------|
| `feat` | Nova funcionalidade |
| `fix` | Correção de bug |
| `docs` | Só documentação |
| `style` | Formatação, sem mudar comportamento |
| `refactor` | Reestrutura sem mudar comportamento |
| `perf` | Melhoria de performance |
| `test` | Adiciona ou corrige testes |
| `chore` | Build, dependências, configuração |

### Regras de ouro

1. **Imperativo, presente**: "adiciona", não "adicionado" nem "adicionando".
   *Teste:* a frase deve completar "Se aplicado, este commit vai ___".
2. **Primeira linha até ~50 caracteres**, sem ponto final.
3. **Linha em branco** antes do corpo.
4. **O corpo explica o PORQUÊ.** O "o quê" já está no diff.

### Comparação

| ❌ Ruim | ✅ Bom |
|---------|--------|
| `ajustes` | `fix: corrige soma de pedidos cancelados no faturamento` |
| `update` | `feat(relatorio): adiciona agrupamento por canal de venda` |
| `WIP` | `refactor: extrai validação de linha para módulo próprio` |
| `corrigido bug do relatorio que estava somando errado os pedidos` | `fix: exclui pedidos cancelados do faturamento` |

In [ ]:
# Commit com corpo: use várias flags -m ou um editor.
# Cada -m vira um parágrafo.
escrever("relatorio_vendas.py", '''"""Relatório de vendas da Aurora Comércio."""

STATUS_FATURAVEL = "pago"


def calcular_faturamento(pedidos):
    """Soma o valor dos pedidos faturáveis."""
    return sum(
        p["quantidade"] * p["preco"]
        for p in pedidos
        if p["status"] == STATUS_FATURAVEL
    )


def agrupar_por_cidade(pedidos):
    """Faturamento por cidade."""
    total = {}
    for p in pedidos:
        if p["status"] != STATUS_FATURAVEL:
            continue
        total[p["cidade"]] = total.get(p["cidade"], 0.0) + p["quantidade"] * p["preco"]
    return total


if __name__ == "__main__":
    print("Atlas — relatório de vendas")
''')

git("add", "relatorio_vendas.py")
git(
    "commit",
    "-m", "feat(relatorio): adiciona agrupamento de faturamento por cidade",
    "-m", "A diretoria precisa saber o desempenho de cada praça para decidir\nonde investir em anúncios. Antes só tínhamos o total consolidado.",
    "-m", "Refs: AURORA-12",
)

In [ ]:
git("log")

## 7. Lendo o histórico com `git log`

O `log` cru é verboso. Estas variações são as que você vai usar de verdade:

| Comando | Para quê |
|---------|----------|
| `git log --oneline` | Uma linha por commit — o mais usado |
| `git log --oneline --graph --all` | Desenha o grafo dos branches |
| `git log -3` | Só os 3 últimos |
| `git log --stat` | Quais arquivos mudaram e quanto |
| `git log -p` | O diff completo de cada commit |
| `git log --author="Maria"` | Filtra por autor |
| `git log --since="2 weeks ago"` | Filtra por data |
| `git log --grep="frete"` | Busca na mensagem |
| `git log -S "calcular_frete"` | Busca no **código** (quando aquela função nasceu?) |
| `git log -- arquivo.py` | História de um arquivo específico |

In [ ]:
git("log", "--oneline")

In [ ]:
git("log", "--stat")

In [ ]:
# Formato customizado — útil para relatórios
git("log", "--pretty=format:%h | %an | %ar | %s")

> 💡 **`git log -S "texto"`** (chamado "pickaxe") é subestimado. Ele encontra os commits em que aquele texto foi **adicionado ou removido** do código. É a forma mais rápida de responder "quando essa função apareceu?" ou "quem apagou essa validação?".

## 8. `git diff` — vendo exatamente o que mudou

| Comando | Compara |
|---------|---------|
| `git diff` | Working directory ↔ **staging** |
| `git diff --staged` | Staging ↔ **último commit** |
| `git diff HEAD` | Working directory ↔ **último commit** |
| `git diff abc123 def456` | Dois commits |
| `git diff main..feature` | Dois branches |
| `git diff -- arquivo.py` | Só um arquivo |

`HEAD` é um apelido para "o commit onde eu estou agora".

In [ ]:
# Vamos alterar o arquivo sem adicionar ao stage
escrever("relatorio_vendas.py", '''"""Relatório de vendas da Aurora Comércio."""

STATUS_FATURAVEL = "pago"
TAXA_IMPOSTO = 0.18


def calcular_faturamento(pedidos):
    """Soma o valor dos pedidos faturáveis."""
    return sum(
        p["quantidade"] * p["preco"]
        for p in pedidos
        if p["status"] == STATUS_FATURAVEL
    )


def agrupar_por_cidade(pedidos):
    """Faturamento por cidade."""
    total = {}
    for p in pedidos:
        if p["status"] != STATUS_FATURAVEL:
            continue
        total[p["cidade"]] = total.get(p["cidade"], 0.0) + p["quantidade"] * p["preco"]
    return total


if __name__ == "__main__":
    print("Atlas — relatório de vendas")
''')

git("diff")

### Como ler um diff

```diff
diff --git a/relatorio_vendas.py b/relatorio_vendas.py
index 8f2a1b3..c4d5e6f 100644
--- a/relatorio_vendas.py          ← 'a' = versão antiga
+++ b/relatorio_vendas.py          ← 'b' = versão nova
@@ -2,6 +2,7 @@                    ← cabeçalho do trecho ("hunk")
                                       -2,6 = na antiga, 6 linhas a partir da 2
                                       +2,7 = na nova, 7 linhas a partir da 2
 STATUS_FATURAVEL = "pago"         ← linha de contexto (sem sinal)
+TAXA_IMPOSTO = 0.18               ← linha ADICIONADA
-LINHA_ANTIGA = 1                  ← linha REMOVIDA
```

Uma linha **modificada** aparece como uma remoção seguida de uma adição — o Git trabalha por linhas, não por caracteres.

In [ ]:
# Depois do add, 'git diff' fica vazio e 'git diff --staged' mostra a mudança
git("add", ".")
print(">>> git diff (working vs staging) — deve estar vazio:")
git("diff")
print(">>> git diff --staged (staging vs último commit):")
git("diff", "--staged")

In [ ]:
git("commit", "-m", "chore: adiciona constante de taxa de imposto")
git("log", "--oneline")

## 9. `.gitignore` — o que **não** versionar

Nem tudo deve ir para o repositório. A regra:

> **Versione o que é fonte. Ignore o que é gerado, secreto ou local.**

### O que ignorar

| Categoria | Exemplos | Por quê |
|-----------|----------|---------|
| **Segredos** | `.env`, `credenciais.json`, `*.pem` | 🔴 Vazamento de senha. Irreversível. |
| **Ambientes** | `.venv/`, `node_modules/` | Reinstalável, pesado |
| **Gerados** | `__pycache__/`, `*.pyc`, `dist/` | Recriado automaticamente |
| **Saídas** | `saida/`, `*.log` | Resultado, não fonte |
| **Do editor** | `.vscode/`, `.idea/` | Preferência pessoal |
| **Do SO** | `.DS_Store`, `Thumbs.db` | Lixo do sistema |
| **Dados pesados** | `*.csv` grandes, `*.db` | Git não é feito para binários grandes |

### Sintaxe

```gitignore
__pycache__/        # pasta (a barra no fim é importante)
*.log               # padrão glob
!importante.log     # exceção: NÃO ignore este
/saida              # só na raiz (a barra no início)
dados/**/temp       # ** = qualquer profundidade
```

In [ ]:
escrever(".gitignore", '''# ── Python ──────────────────────────────────────────
__pycache__/
*.py[cod]
*.egg-info/
.pytest_cache/
.mypy_cache/
.ruff_cache/

# ── Ambientes virtuais ──────────────────────────────
.venv/
venv/
env/

# ── Segredos ────────────────────────────────────────
.env
.env.*
!.env.example
*.pem
credenciais*.json

# ── Saídas geradas ──────────────────────────────────
saida/
*.log

# ── Editor ──────────────────────────────────────────
.vscode/
.idea/
*.swp

# ── Sistema operacional ─────────────────────────────
.DS_Store
Thumbs.db
desktop.ini
''')

# Vamos criar lixo que DEVE ser ignorado
escrever("__pycache__/relatorio.cpython-312.pyc", "bytecode falso")
escrever(".env", "SENHA_BANCO=supersecreta123")
escrever("saida/relatorio.txt", "relatório gerado")
escrever(".env.example", "SENHA_BANCO=coloque-a-senha-aqui")

git("status")

> 📌 Repare: `__pycache__/`, `.env` e `saida/` **não aparecem**. Já `.env.example` aparece, porque a linha `!.env.example` abriu uma exceção. Esse é o padrão profissional: você versiona um arquivo de **exemplo** documentando quais variáveis existem, sem nunca versionar os valores reais.

In [ ]:
git("add", ".")
git("commit", "-m", "chore: adiciona .gitignore e exemplo de variáveis de ambiente")
git("log", "--oneline")

# Confirmando o que está sendo rastreado
print(">>> Arquivos versionados:")
git("ls-files")

### ⚠️ A armadilha nº 1: `.gitignore` não desrastreia

**O `.gitignore` só vale para arquivos que o Git ainda não conhece.** Se você já commitou o `.env`, adicioná-lo ao `.gitignore` depois não faz nada — o Git continua rastreando.

A correção:

```bash
# Remove do índice mas MANTÉM no disco (o --cached é essencial)
git rm --cached .env
git commit -m "chore: remove .env do versionamento"
```

E o mais importante:

> 🔴 **Se um segredo já foi commitado e enviado para o GitHub, considere-o comprometido.** Removê-lo em um commit novo **não apaga o histórico** — qualquer pessoa com acesso ao repositório pode ver o commit antigo. A única resposta correta é **trocar a senha/chave imediatamente**. Reescrever o histórico (com `git filter-repo` ou BFG) é um segundo passo, não o primeiro.

In [ ]:
# Demonstração: arquivo já rastreado que depois entra no .gitignore
escrever("config_local.ini", "[db]\nhost=localhost\nsenha=123")
git("add", "config_local.ini")
git("commit", "-m", "chore: adiciona config local (ops, não devia)")

# Adicionamos ao .gitignore...
conteudo = (LAB / ".gitignore").read_text(encoding="utf-8")
escrever(".gitignore", conteudo + "\nconfig_local.ini\n")

escrever("config_local.ini", "[db]\nhost=localhost\nsenha=456")
print(">>> Mesmo no .gitignore, o Git continua vendo a mudança:")
git("status", "--short")

In [ ]:
# A correção: git rm --cached
git("rm", "--cached", "config_local.ini")
git("add", ".gitignore")
git("commit", "-m", "chore: remove config local do versionamento")

print(">>> Agora sim, ignorado:")
git("status", "--short")
print(">>> O arquivo continua no disco?",
      (LAB / "config_local.ini").exists())

### Templates prontos

Não escreva `.gitignore` do zero. Use:

- [github.com/github/gitignore](https://github.com/github/gitignore) — templates oficiais por linguagem
- [gitignore.io](https://www.toptal.com/developers/gitignore) — gerador combinando linguagem + editor + SO

Para o Atlas, combine: `Python` + `VisualStudioCode` + seu sistema operacional.

## 10. Removendo e renomeando

| Comando | O que faz |
|---------|-----------|
| `git rm arquivo` | Apaga do disco **e** do índice |
| `git rm --cached arquivo` | Tira do índice, **mantém** no disco |
| `git mv antigo novo` | Renomeia e já registra |

Na prática, você pode simplesmente renomear/apagar pelo explorador e rodar `git add -A` — o Git detecta sozinho. `git mv` é só um atalho.

In [ ]:
git("mv", "relatorio_vendas.py", "relatorio.py")
git("status", "--short")
git("commit", "-m", "refactor: renomeia relatorio_vendas.py para relatorio.py")
git("log", "--oneline", "--stat", "-1")

## 🔧 Prática guiada — Versionando o Atlas do zero

Vamos simular a primeira semana do Atlas: vários commits pequenos e coerentes, cada um com uma mensagem que explica a intenção.

In [ ]:
import shutil

# Repositório novo, limpo
ATLAS = Path("lab_git/atlas_demo").resolve()
shutil.rmtree(ATLAS, ignore_errors=True)
ATLAS.mkdir(parents=True)

def g(*args):
    return git(*args, cwd=ATLAS)

def e(nome, conteudo):
    return escrever(nome, conteudo, pasta=ATLAS)

g("init", "-b", "main")
g("config", "user.name", "Aluno Atlas")
g("config", "user.email", "aluno@aurora.com.br")

In [ ]:
# Commit 1 — estrutura mínima e documentação
e("README.md", """# Atlas — Sistema Central da Aurora Comércio

Relatórios de vendas a partir de arquivos CSV.

## Uso

```bash
python main.py dados/vendas.csv
```
""")

e(".gitignore", """__pycache__/
*.pyc
.venv/
.env
saida/
""")

g("add", ".")
g("commit", "-m", "chore: estrutura inicial do projeto com README e .gitignore")

In [ ]:
# Commit 2 — os dados de exemplo
e("dados/vendas.csv", """id,cidade,produto,quantidade,preco,status
1001,Campinas,Notebook,2,2599.90,pago
1002,São Paulo,Mouse,10,89.90,pago
1003,Campinas,Teclado,3,249.00,cancelado
1004,Sorocaba,Monitor,1,1199.00,pago
""")

g("add", "dados/")
g("commit", "-m", "chore(dados): adiciona CSV de exemplo para desenvolvimento")

In [ ]:
# Commit 3 — leitura
e("leitura.py", '''"""Leitura do CSV de vendas."""

import csv
from pathlib import Path


def ler_vendas(caminho: Path) -> list[dict]:
    """Lê o CSV e converte os tipos numéricos."""
    registros = []
    with open(caminho, newline="", encoding="utf-8") as f:
        for linha in csv.DictReader(f):
            linha["quantidade"] = int(linha["quantidade"])
            linha["preco"] = float(linha["preco"])
            registros.append(linha)
    return registros
''')

g("add", "leitura.py")
g("commit", "-m", "feat(leitura): adiciona leitor de CSV com conversão de tipos")

In [ ]:
# Commit 4 — métricas
e("metricas.py", '''"""Cálculo das métricas de vendas."""

from collections import defaultdict

STATUS_FATURAVEL = "pago"


def faturamento_total(vendas: list[dict]) -> float:
    """Soma apenas os pedidos faturáveis."""
    return sum(
        v["quantidade"] * v["preco"]
        for v in vendas
        if v["status"] == STATUS_FATURAVEL
    )


def por_cidade(vendas: list[dict]) -> dict[str, float]:
    """Agrupa o faturamento por cidade."""
    total = defaultdict(float)
    for v in vendas:
        if v["status"] == STATUS_FATURAVEL:
            total[v["cidade"]] += v["quantidade"] * v["preco"]
    return dict(total)
''')

g("add", "metricas.py")
g("commit", "-m", "feat(metricas): adiciona faturamento total e agrupamento por cidade")

In [ ]:
# Commit 5 — o ponto de entrada
e("main.py", '''"""Atlas — ponto de entrada."""

import sys
from pathlib import Path

from leitura import ler_vendas
from metricas import faturamento_total, por_cidade


def main() -> None:
    caminho = Path(sys.argv[1] if len(sys.argv) > 1 else "dados/vendas.csv")
    vendas = ler_vendas(caminho)

    print(f"Faturamento total: R$ {faturamento_total(vendas):,.2f}")
    print()
    for cidade, valor in sorted(por_cidade(vendas).items(), key=lambda kv: -kv[1]):
        print(f"  {cidade:<15} R$ {valor:>12,.2f}")


if __name__ == "__main__":
    main()
''')

g("add", "main.py")
g("commit", "-m", "feat(cli): adiciona ponto de entrada do relatório")

In [ ]:
# O programa funciona?
import subprocess

r = subprocess.run(["python", "main.py"], cwd=ATLAS, capture_output=True, text=True)
print(r.stdout or r.stderr)

In [ ]:
# Commit 6 — a correção que a diretora pediu
e("metricas.py", '''"""Cálculo das métricas de vendas."""

from collections import defaultdict

STATUS_FATURAVEL = "pago"
STATUS_CANCELADO = "cancelado"


def faturamento_total(vendas: list[dict]) -> float:
    """Soma apenas os pedidos faturáveis."""
    return sum(
        v["quantidade"] * v["preco"]
        for v in vendas
        if v["status"] == STATUS_FATURAVEL
    )


def por_cidade(vendas: list[dict]) -> dict[str, float]:
    """Agrupa o faturamento por cidade."""
    total = defaultdict(float)
    for v in vendas:
        if v["status"] == STATUS_FATURAVEL:
            total[v["cidade"]] += v["quantidade"] * v["preco"]
    return dict(total)


def taxa_cancelamento(vendas: list[dict]) -> float:
    """Proporção de pedidos cancelados sobre o total."""
    if not vendas:
        return 0.0
    cancelados = sum(1 for v in vendas if v["status"] == STATUS_CANCELADO)
    return cancelados / len(vendas)
''')

g("add", "metricas.py")
g(
    "commit",
    "-m", "feat(metricas): adiciona taxa de cancelamento",
    "-m", "A diretoria comercial quer acompanhar o percentual de pedidos\ncancelados por período para investigar problemas de estoque.",
)

In [ ]:
# A história do projeto
g("log", "--oneline", "--graph")

In [ ]:
# Quem tocou em cada arquivo, e quando
g("log", "--pretty=format:%h %ad %s", "--date=short", "--", "metricas.py")

> 💡 **Repare no que acabamos de construir.** Seis commits, cada um com uma intenção clara. Se amanhã o cálculo de faturamento der problema, `git log -- metricas.py` mostra exatamente quando aquele arquivo mudou e por quê. Isso é o que substitui `relatorio_final_AGORA_VAI.py`.

## 📝 Exercícios rápidos

Use a função `git(...)` do laboratório ou, melhor ainda, abra o terminal do VS Code e faça de verdade.

**E1.** Crie um repositório novo em `lab_git/exercicio1`, configure `user.name` e `user.email` locais, e faça o primeiro commit com um `README.md`.

**E2.** Crie três arquivos. Adicione **apenas dois** ao stage e commite. Rode `git status` e explique, em um comentário, o estado de cada um dos três.

**E3.** Modifique um arquivo já commitado. Antes de dar `add`, rode `git diff`. Depois do `add`, rode `git diff` e `git diff --staged`. Explique a diferença.

**E4.** Escreva um `.gitignore` que ignore `*.log` **exceto** `importante.log`. Crie os dois arquivos e prove com `git status` que funcionou.

**E5.** Faça 5 commits e depois use `git log --oneline`, `git log --stat` e `git log -p -1`. Qual você usaria para cada situação: "quero uma visão geral", "quero saber quais arquivos mudaram", "quero revisar o código do último commit"?

**E6.** Reescreva estas mensagens no padrão Conventional Commits:
   - `mudei o arquivo`
   - `agora funciona`
   - `add funcao pra calcular frete e tambem arrumei o bug do desconto`

In [ ]:
# E1

In [ ]:
# E2

In [ ]:
# E3

In [ ]:
# E4

In [ ]:
# E5

<!-- E6 — escreva suas respostas aqui -->

## 📋 Cola de referência

```bash
# ── Configuração (uma vez por máquina) ──
git config --global user.name "Seu Nome"
git config --global user.email "seu@email.com"
git config --global init.defaultBranch main
git config --list

# ── Iniciar ──
git init                    # cria repositório na pasta atual
git init -b main            # já nomeando o branch

# ── Ciclo diário ──
git status                  # ONDE ESTOU? (use sempre)
git status --short          # versão compacta
git add arquivo.py          # um arquivo
git add .                   # tudo (cuidado!)
git add -p                  # interativo, pedaço por pedaço
git commit -m "feat: ..."   # grava o snapshot
git commit -am "fix: ..."   # add + commit (só arquivos JÁ rastreados)

# ── Histórico ──
git log --oneline           # o mais usado
git log --oneline --graph --all
git log --stat              # arquivos alterados
git log -p -1               # diff do último commit
git log -S "funcao"         # quando esse código apareceu?
git log -- arquivo.py       # história de um arquivo

# ── Diferenças ──
git diff                    # working vs staging
git diff --staged           # staging vs último commit
git diff HEAD               # working vs último commit

# ── Arquivos ──
git rm arquivo              # remove do disco e do índice
git rm --cached arquivo     # tira do índice, mantém no disco
git mv antigo novo          # renomeia
git ls-files                # o que está sendo rastreado
```

## ✅ Checklist de saída

- [ ] Explico por que o Git guarda snapshots e não diferenças
- [ ] Desenho de cabeça as três áreas e os comandos entre elas
- [ ] Sei para que serve a staging area (e por que ela não é burocracia)
- [ ] Configurei `user.name` e `user.email` globais
- [ ] Uso `git status` reflexivamente, o tempo todo
- [ ] Escrevo mensagens no imperativo, no padrão Conventional Commits
- [ ] Sei ler um diff (`+`, `-`, `@@`, `a/` e `b/`)
- [ ] Sei quais categorias de arquivo nunca devem ser versionadas
- [ ] Sei que `.gitignore` não desrastreia — e conheço `git rm --cached`
- [ ] Entendo que segredo commitado é segredo comprometido

---

### ➡️ Próxima aula

**`02_02_Branches_e_Remotos.ipynb`** — Branches, merge, conflitos e GitHub. Onde você para de trabalhar sozinho na linha do tempo principal.